# Notebook 2: Engenharia e Seleção de Features

**Unidade Curricular:** Dados e Aprendizagem Automática (DAA) — 2026/2027

Este notebook parte dos dados limpos do Notebook 1 e prepara-os para a modelação:

1. **Codificação** das variáveis categóricas em números (requisito dos modelos);
2. **Seleção** de features, removendo colunas sem informação ou redundantes;
3. **Criação** de novas features a partir do conhecimento obtido na análise exploratória, incluindo a **idade mínima** declarada;
4. **Avaliação** do contributo de cada etapa e gravação dos dados finais.

## 1. Configuração e Carregamento dos Dados

Carregamos os dados produzidos pelo Notebook 1, com o `PID` como índice. Na leitura, apenas as células vazias são tratadas como valores em falta (`keep_default_na=False`), para que o pandas não converta textos como "NA" ou "None" em `NaN` por engano.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

DATA_DIR = '../data'
TARGET = 'AGE_RANGE'
RANDOM_STATE = 42
CODIGOS = ['Sem resposta', 'NR', 'Não aplicável']

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), index_col='PID',
                       keep_default_na=False, na_values=[''])
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'), index_col='PID',
                      keep_default_na=False, na_values=[''])

cols_num = train_df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cols_cat = train_df.select_dtypes(include='object').columns.tolist()

print(f"Treino: {train_df.shape} | Teste: {test_df.shape}")
print(f"Numéricas: {len(cols_num)} | Categóricas: {len(cols_cat)}")

Treino: (10962, 261) | Teste: (3655, 260)
Numéricas: 33 | Categóricas: 227


## 2. Inventário das Escalas de Resposta

Muitas perguntas do questionário partilham a mesma escala de resposta (ex.: "Nenhuma / Alguma / Muita dificuldade"). Agrupamos as variáveis categóricas pelo seu conjunto de respostas (sem os códigos "Sem resposta" e "Não aplicável"), para definir a codificação uma vez por escala, e não uma vez por coluna.

In [2]:
escalas = {}
for c in cols_cat:
    respostas = pd.concat([train_df[c], test_df[c]]).unique()
    respostas = tuple(sorted(set(respostas) - set(CODIGOS)))
    escalas.setdefault(respostas, []).append(c)

inventario = pd.DataFrame([{'Nº colunas': len(cols), 'Nº respostas': len(resp),
                            'Respostas': ' | '.join(resp), 'Colunas (exemplo)': ', '.join(cols[:4])}
                           for resp, cols in escalas.items()])
inventario = inventario.sort_values('Nº colunas', ascending=False).reset_index(drop=True)

print(f"{len(cols_cat)} colunas categóricas -> {len(inventario)} escalas diferentes")
with pd.option_context('display.max_colwidth', None):
    display(inventario.head(15))

227 colunas categóricas -> 70 escalas diferentes


,Nº colunas,Nº respostas,Respostas,Colunas (exemplo)
0,78,2,Não | Sim,"IN1, IN6, IN7a, IN7b"
1,29,1,Valor,"IN2_ESTADO, IN3_ESTADO, IN4_ESTADO, HHNBPERS_COD_ESTADO"
2,15,4,Alguma dificuldade | Muita dificuldade | Nenhuma dificuldade | Não consegue de todo,"PL2, PL4, PL5, IN8"
3,8,5,1 a 3 vezes por semana | 4 a 6 vezes por semana | Menos de uma vez por semana | Nunca | Uma vez ou mais por dia,"DH1, DH3, DH5, DH6"
4,8,4,Mais do que metade dos dias | Nunca | Quase todos os dias | Vários dias,"MH1A, MH1B, MH1C, MH1D"
5,7,5,Alguma dificuldade | Muita dificuldade | Nenhuma dificuldade | Não aplicável (nunca tentou ou não precisa de o fazer) | Não consegue de todo,"HA1A, HA1B, HA1C, HA1D"
6,5,7,De acordo | Em desacordo | Mais ou menos de acordo | Mais ou menos em desacordo | Nem de acordo nem em desacordo | Totalmente de acordo | Totalmente em desacordo,"IN50, IN51, IN52, IN53"
7,4,3,Não houve necessidade | Precisou e não satisfez essa necessidade | Precisou e satisfez essa necessidade,"UN2A, UN2B, UN2C, UN2D"
8,3,3,Autóctone (PT) | Outro estado membro da UE | Outro país fora da UE,"BIRTHPLACE_COD, BIRTHPLACEFATH_COD, BIRTHPLACEMOTH_COD"
9,3,5,5 anos ou mais | Entre 1 ano e menos de 3 anos | Entre 3 anos e menos de 5 anos | Nos últimos 12 meses | Nunca,"PA2, PA3, PA4"


## 3. Codificação das Variáveis Categóricas

Os modelos só aceitam números, pelo que cada variável categórica é codificada consoante o seu tipo:

- **Ordinais e binárias** (ex.: "Nenhuma / Alguma / Muita dificuldade", "Não / Sim"): as respostas são substituídas pela sua posição na escala (0, 1, 2, ...), preservando a ordem;
- **Nominais** (sem ordem, ex.: região, situação profissional): *one-hot encoding*, com uma coluna binária por resposta.

Os códigos do inquérito, que não pertencem à escala, recebem valores negativos: **−1** para "Não aplicável", **−2** para "Sem resposta" e **−3** para respostas especiais fora da escala ("Não aplicável (nunca tentou ou não precisa de o fazer)", "Não consegue avaliar"). Mantém-se assim a distinção entre não aplicável e não respondido, essencial por exemplo nas perguntas de autonomia, só feitas a partir dos 55 anos.

### 3.1 Definição das Escalas

Cada escala é definida uma única vez, como uma lista de respostas por ordem crescente. Cada coluna é associada automaticamente à escala que contém todas as suas respostas.

In [3]:
def escala(respostas, especiais=None):
    """Lista ordenada de respostas -> {resposta: 0, 1, 2, ...}, mais respostas especiais."""
    mapa = {r: i for i, r in enumerate(respostas)}
    mapa.update(especiais or {})
    return mapa


CODIGOS_NUM = {'Não aplicável': -1, 'Sem resposta': -2, 'NR': -2}

ESCALAS = [
    # Binárias e variantes
    escala(['Não', 'Sim'], {'Cego (não consegue ver de todo)': 2, 'Surdo (surdez profunda)': 2,
                            'Desloca-se em cadeira de rodas': 2}),
    escala(['Não houve necessidade', 'Não', 'Sim']),
    escala(['Valor', 'Gravidez sem termo (n.a.)']),
    escala(['Homem', 'Mulher']),
    escala(['Sem relação de conjugalidade', 'Com relação de conjugalidade legal ou de facto']),
    escala(['Tempo parcial', 'Tempo completo']),
    escala(['Não utiliza', 'Utiliza']),

    # Dificuldade, limitação, dor, saúde
    escala(['Nenhuma dificuldade', 'Alguma dificuldade', 'Muita dificuldade', 'Não consegue de todo'],
           {'Não aplicável (nunca tentou ou não precisa de o fazer)': -3}),
    escala(['Sozinho, sem dificuldade', 'Sozinho, mas com dificuldade', 'Só com ajuda']),
    escala(['Muito bom', 'Bom', 'Razoável', 'Mau', 'Muito mau']),
    escala(['Nada limitado', 'Limitado mas não severamente', 'Severamente limitado']),
    escala(['Nenhuma', 'Muito ligeira', 'Ligeira', 'Moderada', 'Intensa', 'Muito intensa']),
    escala(['Nada', 'Um pouco', 'De forma moderada', 'Bastante', 'De forma extrema']),
    escala(['Nunca', 'Vários dias', 'Mais do que metade dos dias', 'Quase todos os dias']),
    escala(['Não foi necessária intervenção', 'Intervenção médica ou de enfermeiro',
            'Cuidados em hospital (com ou sem internamento)']),
    escala(['1-7', '8-14', '15-30', '31-180', '>= 181']),
    escala(['Não houve necessidade', 'Precisou e satisfez essa necessidade',
            'Precisou e não satisfez essa necessidade']),

    # Há quanto tempo (do mais recente ao "Nunca")
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 3 anos', 'Entre 3 anos e menos de 5 anos',
            '5 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 2 anos', 'Entre 2 anos e menos de 3 anos',
            'Entre 3 anos e menos de 5 anos', '5 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 2 anos', 'Entre 2 anos e menos de 3 anos',
            '3 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 5 anos', 'Entre 5 anos e menos de 10 anos',
            '10 anos ou mais', 'Nunca']),
    escala(['Há menos de 12 meses', 'Há 12 meses ou mais', 'Nunca']),
    escala(['Há menos de 6 meses', '6 a menos de 12 meses', 'Há 12 meses ou mais', 'Nunca']),
    escala(['Há menos de 10 anos', 'Entre 10 anos e menos de 20 anos', '20 anos ou mais', 'Nunca']),
    escala(['Data', 'Só ano', 'Há mais de 1 ano', 'Nunca']),

    # Frequências e quantidades
    escala(['Nunca', 'Menos de uma vez por semana', '1 a 3 vezes por semana', '4 a 6 vezes por semana',
            'Uma vez ou mais por dia']),
    escala(['Nunca', 'Menos de uma vez por semana', 'Algumas vezes por semana',
            'Todos os dias, apenas antes de se deitar', 'Todos os dias, de manhã e antes de se deitar',
            'Todos os dias, de manhã, após o almoço e antes de se deitar'],
           {'Todos os dias, apenas de manhã': 3}),
    escala(['10 a 29 minutos', '30 a 59 minutos', '1 hora a menos de 2 horas', '2 horas a menos de 3 horas',
            '3 horas ou mais']),
    escala(['Não executa nenhuma das tarefas indicadas',
            'Sentada ou em pé em atividades que envolvem um esforço físico ligeiro',
            'Em movimento ou em tarefas que exigem um esforço físico moderado',
            'Em trabalhos pesados ou fisicamente exigentes']),
    escala(['Menos de 10 horas', 'Entre 10 horas e menos de 20 horas', '20 horas ou mais']),

    # Tabaco e álcool
    escala(['Não fuma', 'Já fumou, mas atualmente não fuma', 'Fuma ocasionalmente', 'Fuma diariamente']),
    escala(['Não fumou', 'Ocasionalmente (menos de uma vez por mês)', 'Ocasionalmente (uma vez por mês)',
            'Ocasionalmente (alguns dias por mês)', 'Ocasionalmente (alguns dias por semana)', 'Diariamente']),
    escala(['Nunca', 'Não, mas já utilizou', 'Sim, ocasionalmente', 'Sim, diariamente']),
    escala(['Nunca ou quase nunca', 'Menos de uma vez por semana',
            'Pelo menos uma vez por semana (mas não diariamente)', 'Diariamente, menos de 1 hora por dia',
            'Diariamente, 1 hora ou mais por dia']),
    escala(['Nunca consumiu, ou só ocasionalmente para provar',
            'Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool',
            'Menos de uma vez por mês', 'Uma vez por mês', '2 a 3 dias por mês', '1 a 2 dias por semana',
            '3 a 4 dias por semana', '5 a 6 dias por semana', 'Todos os dias ou quase todos os dias']),
    escala(['0 bebidas por dia', '1 bebida por dia', '2 bebidas por dia', '3 bebidas por dia',
            '4 a 5 bebidas por dia', '6 a 9 bebidas por dia', '10 a 15 bebidas por dia',
            '16 ou mais bebidas por dia']),
    escala(['Em nenhum dos dias', '1 dia', '2 dias', '3 dias', '4 dias']),

    # Opiniões e suporte social
    escala(['Totalmente em desacordo', 'Em desacordo', 'Mais ou menos em desacordo',
            'Nem de acordo nem em desacordo', 'Mais ou menos de acordo', 'De acordo', 'Totalmente de acordo']),
    escala(['Nenhuma', '1 ou 2', '3 a 5', '6 ou mais']),
    escala(['Nenhuma preocupação e interesse', 'Pouca preocupação e interesse',
            'Alguma preocupação e interesse', 'Muita preocupação e interesse'],
           {'Não consegue avaliar': -3}),
    escala(['Muito difícil', 'Difícil', 'Possível', 'Fácil', 'Muito fácil']),

    # Socioeconómicas
    escala(['Nenhum nível completado', 'Básico 1º e 2º ciclo', 'Básico 3º ciclo', 'Secundário',
            'Pós secundário', 'C.T.Superior Profissional', 'Bacharelato ou licenciatura 1º ciclo',
            'Licenciatura pré-bolonha ou mestrado 2º ciclo', 'Doutoramento']),
    escala(['1º quintil', '2º quintil', '3º quintil', '4º quintil', '5º quintil']),
    escala(['Áreas pouco povoadas', 'Áreas mediamente povoadas', 'Áreas densamente povoadas']),
]

NOMINAIS = ['REGION', 'NUTS2_AG', 'BIRTHPLACE_COD', 'BIRTHPLACEFATH_COD', 'BIRTHPLACEMOTH_COD',
            'CITIZEN_COD', 'MARSTALEGAL', 'MAINSTAT', 'JOBSTAT_COD', 'JOBISCO_COD', 'LOCNACE_COD',
            'HHTYPE', 'IN5', 'IN14', 'IN16', 'IN17', 'IN21', 'IN22', 'IN44', 'IN49', 'IN62', 'IC2']

print(f"{len(ESCALAS)} escalas ordinais/binárias definidas | {len(NOMINAIS)} colunas nominais")

44 escalas ordinais/binárias definidas | 22 colunas nominais


### 3.2 Aplicação da Codificação

A codificação é aplicada ao treino e ao teste em conjunto (sem o alvo), garantindo que ambos ficam com exatamente as mesmas colunas. Não há fuga de informação: apenas se usa a lista de respostas possíveis, nunca o alvo.

Nas variáveis numéricas, os valores em falta passam a **−1**, valor impossível nestas colunas (todas ≥ 0). O motivo da falta (não aplicável ou sem resposta) já está guardado nas colunas `_ESTADO`.

In [4]:
dados = pd.concat([train_df.drop(columns=TARGET), test_df])


def encontrar_escala(c):
    """Devolve a primeira escala que contém todas as respostas da coluna."""
    respostas = set(dados[c].unique()) - set(CODIGOS)
    for mapa in ESCALAS:
        if respostas <= set(mapa):
            return mapa
    return None


# 1) Ordinais e binárias
cols_ordinais = [c for c in cols_cat if c not in NOMINAIS]
sem_escala = [c for c in cols_ordinais if encontrar_escala(c) is None]
assert not sem_escala, f"Colunas sem escala definida: {sem_escala}"

for c in cols_ordinais:
    dados[c] = dados[c].map({**encontrar_escala(c), **CODIGOS_NUM})
assert dados[cols_ordinais].isna().sum().sum() == 0, "Há respostas que não estão em nenhuma escala"

# 2) Nominais -> one-hot
dados = pd.get_dummies(dados, columns=NOMINAIS, dtype=int)

# 3) Numéricas: valor em falta -> -1
dados[cols_num] = dados[cols_num].fillna(-1)

# Separar de novo treino e teste (pelo PID)
train_enc = dados.loc[train_df.index].copy()
train_enc[TARGET] = train_df[TARGET]
test_enc = dados.loc[test_df.index].copy()

print(f"Ordinais/binárias: {len(cols_ordinais)} | Nominais: {len(NOMINAIS)} -> "
      f"{dados.shape[1] - len(cols_ordinais) - len(cols_num)} colunas one-hot")
print(f"Treino: {train_enc.shape} | Teste: {test_enc.shape}")
print(f"Colunas não numéricas restantes: {dados.select_dtypes(exclude='number').shape[1]}")
print(f"Valores em falta restantes: {dados.isna().sum().sum()}")

Ordinais/binárias: 205 | Nominais: 22 -> 171 colunas one-hot
Treino: (10962, 410) | Teste: (3655, 409)
Colunas não numéricas restantes: 0
Valores em falta restantes: 0


> **Conclusão:** as 205 variáveis ordinais e binárias foram codificadas numa única coluna cada, preservando a ordem das respostas; as 22 nominais deram origem a 171 colunas *one-hot*. O conjunto tem 409 features, todas numéricas e sem valores em falta.

## 4. Seleção de Features

Removemos colunas que não acrescentam informação:

1. **Quase constantes**: o valor mais frequente ocorre em 99% ou mais das linhas, pelo que a coluna praticamente não distingue pessoas;
2. **Duplicadas**: colunas idênticas a outra em todas as linhas.

O impacto desta etapa no desempenho é avaliado na secção 7, em conjunto com as restantes.

In [5]:
y = train_enc[TARGET]
X_base = train_enc.drop(columns=TARGET)       # dados apenas codificados (para comparar na secção 7)
todas = pd.concat([X_base, test_enc])

# 1) Quase constantes: % do valor mais frequente
freq_max = todas.apply(lambda s: s.value_counts(normalize=True).iloc[0])
quase_constantes = freq_max[freq_max >= 0.99].index.tolist()

# 2) Duplicadas (entre as restantes)
restantes = todas.drop(columns=quase_constantes)
duplicadas = restantes.columns[restantes.T.duplicated()].tolist()

remover = quase_constantes + duplicadas
X_sel = X_base.drop(columns=remover)
test_sel = test_enc.drop(columns=remover)

print(f"Quase constantes: {len(quase_constantes)} | Duplicadas: {len(duplicadas)}")
print(f"Exemplos de quase constantes: {quase_constantes[:10]}")
print(f"Duplicadas: {duplicadas}")
print(f"Colunas: {X_base.shape[1]} -> {X_sel.shape[1]}")

Quase constantes: 70 | Duplicadas: 6
Exemplos de quase constantes: ['AC1A', 'IN13', 'HHNBPERS_COD_ESTADO', 'HHNBPERS_0_13_COD_ESTADO', 'HO12_COD_ESTADO', 'IN29_ESTADO', 'BIRTHPLACE_COD_Sem resposta', 'BIRTHPLACEFATH_COD_Outro estado membro da UE', 'BIRTHPLACEFATH_COD_Sem resposta', 'BIRTHPLACEMOTH_COD_Outro estado membro da UE']
Duplicadas: ['NUTS2_AG_Algarve', 'NUTS2_AG_Norte', 'NUTS2_AG_RA Açores', 'NUTS2_AG_RA Madeira', 'JOBISCO_COD_Não aplicável', 'LOCNACE_COD_Não aplicável']
Colunas: 409 -> 333


> **Conclusão:** foram removidas 70 colunas quase constantes (sobretudo categorias raras do *one-hot* e perguntas quase sempre "Não aplicável") e 6 duplicadas (ex.: as regiões da `NUTS2_AG` que coincidem com as da `REGION`, e as categorias "Não aplicável" da profissão e do setor de atividade, que identificam as mesmas pessoas). O conjunto passou de 409 para 333 colunas.

## 5. Criação de Novas Features

Com base na análise exploratória, criamos quatro features que resumem informação dispersa por várias colunas:

- **`N_DIFICULDADES`**: número de atividades do dia a dia (`HA1*`, `PC1*`) realizadas com dificuldade (alguma, muita ou não consegue), que se acumulam com a idade;
- **`N_DOENCAS`**: número de doenças crónicas declaradas (`CD1A`–`CD1P`), cuja prevalência aumenta com a idade;
- **`IMC`**: índice de massa corporal, peso (kg) / altura (m)², com o valor −1 quando falta o peso ou a altura;
- **`N_NAO_APLICAVEL`**: número de perguntas com resposta "Não aplicável", que resume o percurso da pessoa no questionário.

Verificamos a sua relação com a idade através da correlação de Spearman; o impacto no desempenho é avaliado na secção 7.

In [6]:
cols_ordinais_sel = [c for c in cols_ordinais if c in X_sel.columns]


def criar_features(df):
    novas = pd.DataFrame(index=df.index)

    autonomia = [c for c in df.columns if c.startswith(('HA1', 'PC1'))]
    novas['N_DIFICULDADES'] = (df[autonomia] >= 1).sum(axis=1)

    doencas = [c for c in df.columns if c.startswith('CD1')]
    novas['N_DOENCAS'] = (df[doencas] == 1).sum(axis=1)

    valido = (df['BM1'] > 0) & (df['BM2'] > 0)
    novas['IMC'] = np.where(valido, df['BM2'] / (df['BM1'] / 100) ** 2, -1).round(1)

    novas['N_NAO_APLICAVEL'] = (df[cols_ordinais_sel] == -1).sum(axis=1)
    return novas


novas_treino = criar_features(X_sel)
novas_teste = criar_features(test_sel)

print("Correlação de Spearman com a idade:")
print(novas_treino.join(y).corr(method='spearman')[TARGET].drop(TARGET).round(2))

Correlação de Spearman com a idade:
N_DIFICULDADES     0.58
N_DOENCAS          0.53
IMC                0.18
N_NAO_APLICAVEL   -0.46
Name: AGE_RANGE, dtype: float64


> **Conclusão:** três das novas features estão fortemente relacionadas com a idade: `N_DIFICULDADES` (ρ = 0,58), `N_DOENCAS` (ρ = 0,53) e `N_NAO_APLICAVEL` (ρ = −0,46, já que as pessoas mais novas saltam mais perguntas, como as de autonomia). O `IMC` tem uma relação mais fraca (ρ = 0,18), em linha com o observado no Notebook 1: a obesidade atinge o máximo entre os 55 e os 74 anos, sem crescer de forma contínua com a idade.

## 6. Idade Mínima Declarada

Algumas variáveis parecem corresponder a idades declaradas (ex.: idade em que deixou de fumar). Se assim for, constituem um **limite inferior** para a idade atual: quem deixou de fumar aos 52 anos tem, pelo menos, 52 anos.

Para confirmar quais das variáveis candidatas se comportam como idades, verificamos no treino a percentagem de casos em que o valor ultrapassa o limite superior da faixa etária da pessoa: numa variável de idade, esta percentagem deve ser próxima de 0%. Incluímos ainda a `IN24` (anos desde a última gravidez) somada de 15 anos, como idade mínima para uma gravidez.

In [7]:
LIMITE_SUPERIOR = np.array([17, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84, 120])
idade_max = pd.Series(LIMITE_SUPERIOR[y.values], index=y.index)


def candidatas_idade(df):
    """Variáveis candidatas a idade declarada (-1 = sem valor -> NaN)."""
    cand = pd.DataFrame(index=df.index)
    for c in ['IN2', 'IN3', 'IN4', 'IN25', 'IN26', 'IN45', 'IN47']:
        cand[c] = df[c].where(df[c] >= 0)
    cand['IN24 + 15'] = df['IN24'].where(df['IN24'] >= 0) + 15
    return cand


cand_treino = candidatas_idade(X_sel)

verificacao = []
for c in cand_treino.columns:
    tem_valor = cand_treino[c].notna()
    acima = cand_treino.loc[tem_valor, c] > idade_max[tem_valor]
    verificacao.append({'Variável': c, 'Nº com valor': tem_valor.sum(),
                        'Nº acima da idade máxima': acima.sum(),
                        '% acima da idade máxima': round(acima.mean() * 100, 1)})
display(pd.DataFrame(verificacao))

,Variável,Nº com valor,Nº acima da idade máxima,% acima da idade máxima
0,IN2,1433,0,0.0
1,IN3,1273,0,0.0
2,IN4,746,0,0.0
3,IN25,1000,28,2.8
4,IN26,1334,1,0.1
5,IN45,2232,0,0.0
6,IN47,3273,0,0.0
7,IN24 + 15,1476,3,0.2


> **Conclusão:** as variáveis `IN2`, `IN3`, `IN4`, `IN45` e `IN47` nunca ultrapassam o limite superior da faixa etária da pessoa (0 casos), comportando-se como idades declaradas. A `IN26` (1 caso) e a `IN24 + 15` (3 casos) têm exceções isoladas, consistentes com anomalias de registo, e são igualmente aceites. A `IN25` é excluída (2,8% de violações), não correspondendo a uma idade.
>
> Independentemente do significado exato de cada variável, todas as aceites constituem um **limite inferior válido** para a idade atual.

### 6.1 Construção da Feature

Para cada pessoa, a **`IDADE_MINIMA`** é o maior valor entre as variáveis aceites (15 anos, a idade mínima do inquérito, quando não há nenhum), e a **`FAIXA_MINIMA`** é a faixa etária correspondente. Por resultar do máximo entre várias colunas, esta feature contém informação que um modelo baseado em árvores, que analisa uma variável de cada vez, não consegue construir por si só.

In [8]:
LIMITE_INFERIOR = np.array([15, 18, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85])
VARS_IDADE = ['IN2', 'IN3', 'IN4', 'IN26', 'IN45', 'IN47', 'IN24 + 15']


def features_idade(df):
    idade_min = candidatas_idade(df)[VARS_IDADE].max(axis=1).fillna(15).clip(lower=15)
    faixa_min = np.searchsorted(LIMITE_INFERIOR, idade_min, side='right') - 1
    return pd.DataFrame({'IDADE_MINIMA': idade_min, 'FAIXA_MINIMA': faixa_min}, index=df.index)


idade_treino = features_idade(X_sel)
idade_teste = features_idade(test_sel)

com_info = idade_treino['IDADE_MINIMA'] > 15
print(f"Pessoas com idade mínima informativa: {com_info.sum()} ({com_info.mean():.1%})")
print(f"Casos com faixa mínima acima da faixa real: {(idade_treino['FAIXA_MINIMA'] > y).sum()}")

Pessoas com idade mínima informativa: 5270 (48.1%)
Casos com faixa mínima acima da faixa real: 4


> **Conclusão:** 48,1% das pessoas têm uma idade mínima informativa (acima de 15 anos). Apenas 4 pessoas têm uma faixa mínima superior à sua faixa real; estes casos incoerentes serão analisados no Notebook 4, no diagnóstico de anomalias.

## 7. Contributo de Cada Etapa

Avaliamos o efeito de cada etapa deste notebook com o mesmo modelo e a mesma validação: o Gradient Boosting otimizado no Notebook 3 (secção 4), com os hiperparâmetros aí otimizados, e validação cruzada estratificada em 5 partes. Cada linha acrescenta uma etapa à anterior.

Nota: como os hiperparâmetros foram otimizados sobre os dados finais, a última etapa pode estar ligeiramente favorecida; a tabela serve sobretudo para comparar o contributo relativo de cada etapa.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Melhor configuração encontrada no Notebook 3
MELHORES_PARAMS_GB = {'learning_rate': 0.05, 'max_leaf_nodes': 15,
                      'min_samples_leaf': 20, 'l2_regularization': 1}


def avaliar_gb(X, y):
    """Macro F1 (média e desvio padrão) do Gradient Boosting em validação cruzada."""
    gb = HistGradientBoostingClassifier(max_iter=500, early_stopping=True, validation_fraction=0.1,
                                        n_iter_no_change=20, class_weight='balanced',
                                        random_state=RANDOM_STATE, **MELHORES_PARAMS_GB)
    notas = cross_val_score(gb, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return notas.mean(), notas.std()


etapas = {
    '1. Apenas codificação (secção 3)': X_base,
    '2. + Seleção de features (secção 4)': X_sel,
    '3. + Novas features (secção 5)': X_sel.join(novas_treino),
    '4. + Idade mínima (secção 6)': X_sel.join(novas_treino).join(idade_treino),
}

contributo = []
for nome, X_etapa in etapas.items():
    media, desvio = avaliar_gb(X_etapa, y)
    contributo.append({'Etapa': nome, 'Nº colunas': X_etapa.shape[1],
                       'Macro F1': round(media, 4), 'Desvio padrão': round(desvio, 4)})
    print(f"{nome}: concluído")

display(pd.DataFrame(contributo))

1. Apenas codificação (secção 3): concluído
2. + Seleção de features (secção 4): concluído
3. + Novas features (secção 5): concluído
4. + Idade mínima (secção 6): concluído


,Etapa,Nº colunas,Macro F1,Desvio padrão
0,1. Apenas codificação (secção 3),409,0.4625,0.0051
1,2. + Seleção de features (secção 4),333,0.4688,0.0035
2,3. + Novas features (secção 5),337,0.4663,0.0085
3,4. + Idade mínima (secção 6),339,0.4742,0.0073


> **Conclusão:** a seleção de features melhora o desempenho (0,4625 → 0,4688), indicando que as colunas quase constantes funcionavam como ruído. As novas features da secção 5 não trazem ganho ao Gradient Boosting (0,4663): o modelo consegue aproximar contagens por si só. A **idade mínima** é a etapa com maior contributo (0,4742): por resultar do máximo entre várias colunas, fornece informação que o modelo não consegue construir.
>
> As diferenças entre etapas são da ordem do desvio padrão (~0,006), pelo que devem ser lidas como tendências. Mantêm-se todas as etapas: nenhuma prejudica o resultado de forma clara e o conjunto final obtém o melhor valor.

## 8. Gravação dos Dados Finais

Guardamos a versão final dos dados (codificação, seleção, novas features e idade mínima), que será usada nos Notebooks 3 e 4.

In [10]:
train_final = X_sel.join(novas_treino).join(idade_treino)
train_final[TARGET] = y
test_final = test_sel.join(novas_teste).join(idade_teste)

assert list(train_final.columns.drop(TARGET)) == list(test_final.columns)
assert train_final.isna().sum().sum() == 0 and test_final.isna().sum().sum() == 0

train_final.to_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
test_final.to_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
print(f"Guardado: train_nb02.csv {train_final.shape} | test_nb02.csv {test_final.shape}")

Guardado: train_nb02.csv (10962, 340) | test_nb02.csv (3655, 339)


## 9. Conclusões

- **Codificação:** as 227 variáveis categóricas foram agrupadas em 70 escalas de resposta. As ordinais e binárias foram codificadas pela sua ordem natural (205 colunas) e as nominais por *one-hot* (171 colunas). Os códigos do inquérito receberam valores negativos próprios, preservando a distinção entre "Não aplicável" e "Sem resposta".
- **Seleção:** foram removidas 70 colunas quase constantes e 6 duplicadas (409 → 333), com melhoria do desempenho.
- **Novas features:** contagens de dificuldades, doenças e respostas "Não aplicável", e o IMC. Estão relacionadas com a idade, mas não beneficiam o Gradient Boosting.
- **Idade mínima:** construída a partir de variáveis validadas como idades declaradas; é a feature com maior contributo para o Gradient Boosting.

O conjunto final tem **339 features**, todas numéricas e sem valores em falta.